# 05 - Submission Smoke Test

**Goal:** Validate the submission pipeline quickly without aiming for performance.

**Autograder contract (must match exactly):**

- Submit one zip named `TEAMNAME_AGENT.zip`.
- The zip must extract to one top-level folder named `TEAMNAME_AGENT/`.
- That folder must include `__init__.py` and your runtime agent implementation (for example `agent.py` or `agent_ray.py`).
- Include any model/checkpoint/runtime files your agent imports at inference time.
- Do not submit the entire repository and do not zip loose `.py` files without the folder.

**What you will learn:** How a tiny PPO run becomes a checkpoint, exported package, final zip, importable agent, and structurally ready artifact.

**Inputs:** A working `soccertwos` environment with Ray/RLlib installed.

**Outputs:** `submission_smoke_agent/`, `submission_smoke_agent.zip`, `TEAMNAME_AGENT.zip`, validation tables, and a readiness summary.

**Success criteria:** The final printed summary says the artifact is structurally ready, or names the missing step.

In [1]:
from pathlib import Path
import importlib
import os
import sys

PROJECT_MARKER = Path("soccer_twos_project") / "notebook_tools.py"


def _running_in_colab():
    if "google.colab" in sys.modules:
        return True
    if os.environ.get("COLAB_RELEASE_TAG") or os.environ.get("COLAB_GPU"):
        return True
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _candidate_project_roots():
    seen = set()

    def add(path):
        path = Path(path).expanduser()
        key = str(path)
        if key not in seen:
            seen.add(key)
            yield path

    for env_name in ("SOCCER_TWOS_PROJECT_ROOT", "PROJECT_ROOT"):
        value = os.environ.get(env_name)
        if value:
            yield from add(value)

    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        yield from add(base)
        yield from add(base / "soccer-twos-starter")
        yield from add(base / "project" / "soccer-twos-starter")

    if sys.platform == "darwin":
        yield from add(
            Path.home()
            / "all_data"
            / "Georgia Tech"
            / "Course Content"
            / "CS 8803- DRL"
            / "project"
            / "soccer-twos-starter"
        )

    if _running_in_colab():
        try:
            from google.colab import drive  # type: ignore
            if not Path("/content/drive/MyDrive").exists():
                drive.mount("/content/drive")
        except Exception:
            pass
        for drive_root in (Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives"), Path("/content")):
            for relative in (
                Path("CS 8803- DRL") / "project" / "soccer-twos-starter",
                Path("project") / "soccer-twos-starter",
                Path("soccer-twos-starter"),
                Path("Colab Notebooks") / "soccer-twos-starter",
            ):
                yield from add(drive_root / relative)


def _find_project_root():
    for candidate in _candidate_project_roots():
        if (candidate / PROJECT_MARKER).exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find soccer_twos_project/notebook_tools.py. "
        "Open this notebook from the project root/notebooks folder, or set SOCCER_TWOS_PROJECT_ROOT."
    )


PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for _module_name in list(sys.modules):
    if _module_name == "soccer_twos_project" or _module_name.startswith("soccer_twos_project."):
        del sys.modules[_module_name]

importlib.invalidate_caches()
from IPython.display import Markdown, display
from soccer_twos_project.notebook_tools import *

ctx = setup_project()
show_hardware()

Runtime: linux
Project root: /coc/scratch/vchopra/rl_training/project/soccer-twos-starter
Artifact root: /coc/scratch/vchopra/rl_training/project/soccer-twos-starter/artifacts/cs8803_soccer_twos
Python: /coc/scratch/dgarg/miniconda3/envs/soccertwos/bin/python
soccer_twos: /coc/scratch/dgarg/miniconda3/envs/soccertwos/lib/python3.8/site-packages/soccer_twos/__init__.py
ray: 1.13.0
torch: 1.8.1+cu102
python: /coc/scratch/dgarg/miniconda3/envs/soccertwos/bin/python


/coc/scratch/dgarg/miniconda3/envs/soccertwos/lib/python3.8/site-packages/torch/cuda/__init__.py:104: UserWarning: 
NVIDIA A40 with CUDA capability sm_86 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_37 sm_50 sm_60 sm_70.
If you want to use the NVIDIA A40 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(incompatible_device_warn.format(device_name, capability, " ".join(arch_list), device_name))


{
  "cpu_count": 128,
  "gpu_name": "NVIDIA A40",
  "mlx_available": false,
  "ram_gb": 503.69,
  "torch_cuda_available": true,
  "torch_cuda_device_count": 8,
  "torch_cuda_device_names": [
    "NVIDIA A40",
    "NVIDIA A40",
    "NVIDIA A40",
    "NVIDIA A40",
    "NVIDIA A40",
    "NVIDIA A40",
    "NVIDIA A40",
    "NVIDIA A40"
  ],
  "torch_cuda_runtime_probe_device": "NVIDIA A40",
  "torch_cuda_runtime_probe_error": "RuntimeError: CUDA error: no kernel image is available for execution on the device",
  "torch_cuda_runtime_ready": false,
  "torch_cuda_version": "10.2",
  "torch_mps_available": false,
  "torch_version": "1.8.1+cu102"
}


## Configuration

This notebook is deliberately small. It is not a performance run and should not be used as the final trained agent.

In [2]:
SOURCE_STAGE = "ppo_baseline"
SMOKE_AGENT_NAME = "submission_smoke_agent"

# Set your team token without the _AGENT suffix. The notebook will create TEAMNAME_AGENT/.
TEAM_AGENT_NAME = "test_1"
SMOKE_TIMESTEPS = 3_000

RUN_SMOKE_TRAINING = True
RUN_EXPORT = True
RUN_TINY_ROLLOUT = True
RUN_OPTIONAL_EVAL = False

## 1. Small PPO Smoke Training

This creates a tiny checkpoint. If training fails, the later cells will explain what artifact is missing.

In [3]:
smoke_checkpoint = None
if RUN_SMOKE_TRAINING:
    try:
        smoke_checkpoint = run_training(
            ctx,
            SOURCE_STAGE,
            profile_name="cpu_debug",
            timesteps=SMOKE_TIMESTEPS,
            smoke=True,
            checkpoint_freq=1,
            verbose=1,
        )
    except Exception as exc:
        print("Smoke training failed:", type(exc).__name__, exc)
        print("Activate the soccertwos environment and rerun this notebook.")
else:
    print("RUN_SMOKE_TRAINING=False. Looking for an existing checkpoint.")

if smoke_checkpoint is None:
    try:
        smoke_checkpoint = best_checkpoint(ctx, SOURCE_STAGE)
        print("Using existing checkpoint:", smoke_checkpoint)
    except Exception as exc:
        print("No checkpoint available:", type(exc).__name__, exc)

smoke_checkpoint

2026-04-23 02:20:26,103	INFO tune.py:747 -- Total run time: 26.88 seconds (25.90 seconds for the tuning loop).


Best trial: PPO_Soccer_748ab_00000
Best checkpoint: /coc/scratch/vchopra/rl_training/project/soccer-twos-starter/artifacts/cs8803_soccer_twos/checkpoints/soccer_ppo_baseline/PPO_Soccer_748ab_00000_0_2026-04-23_02-19-59/checkpoint_000001/checkpoint-1
Wrote run metadata: /coc/scratch/vchopra/rl_training/project/soccer-twos-starter/artifacts/cs8803_soccer_twos/checkpoints/soccer_ppo_baseline/run_metadata.json
Returned checkpoint: /coc/scratch/vchopra/rl_training/project/soccer-twos-starter/artifacts/cs8803_soccer_twos/checkpoints/soccer_ppo_baseline/PPO_Soccer_748ab_00000_0_2026-04-23_02-19-59/checkpoint_000001/checkpoint-1


'/coc/scratch/vchopra/rl_training/project/soccer-twos-starter/artifacts/cs8803_soccer_twos/checkpoints/soccer_ppo_baseline/PPO_Soccer_748ab_00000_0_2026-04-23_02-19-59/checkpoint_000001/checkpoint-1'

## 2. Checkpoint And Progress Summary

A structurally valid path needs an actual checkpoint before export.

In [4]:
display(progress_status(ctx, SOURCE_STAGE, rows=5))
print_json(checkpoint_summary(ctx, SOURCE_STAGE))

Latest progress file: /coc/scratch/vchopra/rl_training/project/soccer-twos-starter/artifacts/cs8803_soccer_twos/checkpoints/soccer_ppo_baseline/PPO_Soccer_748ab_00000_0_2026-04-23_02-19-59/progress.csv


,training_iteration,timesteps_total,episodes_total,episode_reward_mean,episode_reward_min,episode_reward_max,episode_len_mean,time_total_s
0,1,1000,1,0.0,0.0,0.0,999.000000,8.131413
1,2,2000,2,0.0,0.0,0.0,999.500000,15.033078
2,3,3000,3,0.0,0.0,0.0,999.666667,21.432186


{
  "checkpoint": "/coc/scratch/vchopra/rl_training/project/soccer-twos-starter/artifacts/cs8803_soccer_twos/checkpoints/soccer_ppo_baseline/PPO_Soccer_748ab_00000_0_2026-04-23_02-19-59/checkpoint_000001/checkpoint-1",
  "criterion": "policy performance baseline",
  "exists": true,
  "experiment": "Soccer",
  "stage": "ppo_baseline",
  "stop": {
    "time_total_s": 1800,
    "timesteps_total": 3000
  }
}


## 3. Export Standalone Agent Package

Export writes `agent.py`, `model.py`, `checkpoint.pth`, metadata, README, requirements, and a zip.

In [5]:
exported_package_dir = ctx.submissions_dir / SMOKE_AGENT_NAME
smoke_package_zip = exported_package_dir.with_suffix(".zip")

if RUN_EXPORT and smoke_checkpoint:
    try:
        from soccer_twos_project.exporting import export_checkpoint
        export_checkpoint(SimpleNamespace(
            checkpoint=smoke_checkpoint,
            stage=SOURCE_STAGE,
            policy_id="default_policy",
            profile="cpu_debug",
            artifact_root=str(ctx.artifact_root),
            output_dir=None,
            agent_name=SMOKE_AGENT_NAME,
            author="Your Name",
            email="your.email@gatech.edu",
            description="Tiny PPO smoke-test export. This validates the submission path, not performance.",
            no_zip=False,
            clean=True,
        ))
    except Exception as exc:
        print("Export failed:", type(exc).__name__, exc)
elif not smoke_checkpoint:
    print("Export skipped because no checkpoint exists.")
else:
    print("Export skipped by RUN_EXPORT=False.")

print_json(package_summary(exported_package_dir) if exported_package_dir.exists() else {"package_dir": str(exported_package_dir), "exists": False})

2026-04-23 02:20:34,195	INFO ppo.py:414 -- In multi-agent mode, policies will be optimized sequentially by the multi-GPU optimizer. Consider setting simple_optimizer=True if this doesn't work for you.


[INFO] Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


INFO:mlagents_envs.environment:Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


[INFO] Connected new brain: SoccerTwos?team=1


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=1


[INFO] Connected new brain: SoccerTwos?team=0


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=0
2026-04-23 02:20:35,299	WARNING env.py:135 -- Your env doesn't have a .spec.max_episode_steps attribute. This is fine if you have set 'horizon' in your config dictionary, or `soft_horizon`. However, if you haven't, 'horizon' will default to infinity, and your environment will not be reset.
2026-04-23 02:20:35,323	INFO ppo_tf_policy.py:382 -- `vf_share_layers=True` in your model. Therefore, remember to tune the value of `vf_loss_coeff`!
2026-04-23 02:20:35,336	INFO torch_policy.py:190 -- TorchPolicy (worker=local) running on CPU.
2026-04-23 02:20:35,349	INFO rollout_worker.py:1793 -- Built policy map: {}
2026-04-23 02:20:35,350	INFO rollout_worker.py:1794 -- Built preprocessor map: {'default_policy': <ray.rllib.models.preprocessors.NoPreprocessor object at 0x76fb12bbcbe0>}
2026-04-23 02:20:35,350	INFO rollout_worker.py:670 -- Built filter map: {'default_policy': <ray.rllib.utils.filter.NoFilter object at 0x76fb12b6707

Wrote package: /coc/scratch/vchopra/rl_training/project/soccer-twos-starter/artifacts/cs8803_soccer_twos/submissions/submission_smoke_agent
Wrote zip: /coc/scratch/vchopra/rl_training/project/soccer-twos-starter/artifacts/cs8803_soccer_twos/submissions/submission_smoke_agent.zip
{
  "exists": true,
  "file_count": 7,
  "files": [
    "README.md",
    "__init__.py",
    "agent.py",
    "checkpoint.pth",
    "metadata.json",
    "model.py",
    "requirements.txt"
  ],
  "has_agent_implementation": true,
  "missing_required_files": [],
  "package_dir": "/coc/scratch/vchopra/rl_training/project/soccer-twos-starter/artifacts/cs8803_soccer_twos/submissions/submission_smoke_agent",
  "required_files_present": [
    "__init__.py"
  ]
}


## 4. Validate Exported Package And Create TEAMNAME_AGENT.zip

This copies the smoke package to `TEAMNAME_AGENT/` and zips it so the archive shape matches submission requirements.

Equivalent manual command (run from the parent directory of `TEAMNAME_AGENT/`):

`zip -r TEAMNAME_AGENT.zip TEAMNAME_AGENT`

In [6]:
final_zip = None
if exported_package_dir.exists():
    try:
        validate_exported_package(ctx, SMOKE_AGENT_NAME, rollout_steps=0)
        final_zip = make_final_submission(ctx, SMOKE_AGENT_NAME, TEAM_AGENT_NAME)
        final_folder = Path(final_zip).with_suffix("").name
        validate_package_folder(ctx.submissions_dir / final_folder)
        validate_zip_manifest(final_zip, expected_top_level=final_folder)
        print("Manual zip command equivalent:")
        print(f"cd {ctx.submissions_dir} && zip -r {Path(final_zip).name} {final_folder}")
    except Exception as exc:
        print("Package/final zip validation failed:", type(exc).__name__, exc)
else:
    print("Final zip skipped because the exported package folder does not exist.")

final_zip

Using Unity base_port: 50041
[INFO] Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


INFO:mlagents_envs.environment:Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


[INFO] Connected new brain: SoccerTwos?team=1


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=1


[INFO] Connected new brain: SoccerTwos?team=0


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=0


Validated package: submission_smoke_agent
Actions: {0: [0, 1, 1], 1: [1, 0, 1]}
{
  "actions": {
    "0": [
      0,
      1,
      1
    ],
    "1": [
      1,
      0,
      1
    ]
  },
  "module_name": "submission_smoke_agent",
  "package": {
    "exists": true,
    "file_count": 10,
    "files": [
      "README.md",
      "__init__.py",
      "__pycache__/__init__.cpython-38.pyc",
      "__pycache__/agent.cpython-38.pyc",
      "__pycache__/model.cpython-38.pyc",
      "agent.py",
      "checkpoint.pth",
      "metadata.json",
      "model.py",
      "requirements.txt"
    ],
    "has_agent_implementation": true,
    "missing_required_files": [],
    "package_dir": "/coc/scratch/vchopra/rl_training/project/soccer-twos-starter/artifacts/cs8803_soccer_twos/submissions/submission_smoke_agent",
    "required_files_present": [
      "__init__.py"
    ]
  }
}
Final submission folder: /coc/scratch/vchopra/rl_training/project/soccer-twos-starter/artifacts/cs8803_soccer_twos/submissions/te

PosixPath('/coc/scratch/vchopra/rl_training/project/soccer-twos-starter/artifacts/cs8803_soccer_twos/submissions/test_1_AGENT.zip')

## 5. Fresh Zip Import And `act()` Test

This unpacks the zip into a temporary folder, imports the agent fresh, instantiates it with a real environment, and calls `act()`.

In [7]:
if final_zip:
    try:
        expected_folder = Path(final_zip).with_suffix("").name
        zip_validation = validate_zip_package(final_zip, expected_top_level=expected_folder)
        print_json(zip_validation)
    except Exception as exc:
        print("Fresh zip validation failed:", type(exc).__name__, exc)
else:
    print("Fresh zip validation skipped because final_zip is missing.")

{
  "file_count": 10,
  "missing_required_files": [],
  "top_level": "test_1_AGENT",
  "zip_path": "/coc/scratch/vchopra/rl_training/project/soccer-twos-starter/artifacts/cs8803_soccer_twos/submissions/test_1_AGENT.zip"
}
Using Unity base_port: 50042
[INFO] Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


INFO:mlagents_envs.environment:Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


[INFO] Connected new brain: SoccerTwos?team=1


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=1


[INFO] Connected new brain: SoccerTwos?team=0


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=0


Validated package: test_1_AGENT
Actions: {0: [0, 1, 1], 1: [1, 0, 1]}
{
  "actions": {
    "0": [
      0,
      1,
      1
    ],
    "1": [
      1,
      0,
      1
    ]
  },
  "manifest": {
    "file_count": 10,
    "files": [
      "test_1_AGENT/model.py",
      "test_1_AGENT/agent.py",
      "test_1_AGENT/__init__.py",
      "test_1_AGENT/checkpoint.pth",
      "test_1_AGENT/metadata.json",
      "test_1_AGENT/requirements.txt",
      "test_1_AGENT/README.md",
      "test_1_AGENT/__pycache__/agent.cpython-38.pyc",
      "test_1_AGENT/__pycache__/model.cpython-38.pyc",
      "test_1_AGENT/__pycache__/__init__.cpython-38.pyc"
    ],
    "missing_required_files": [],
    "required_files": [
      "__init__.py"
    ],
    "top_level": "test_1_AGENT",
    "zip_path": "/coc/scratch/vchopra/rl_training/project/soccer-twos-starter/artifacts/cs8803_soccer_twos/submissions/test_1_AGENT.zip"
  }
}


## 6. Optional Tiny Rollout Watch/Eval

This is still structural. It checks that the copied final package can act for several steps.

In [8]:
smoke_rollout = None
if RUN_TINY_ROLLOUT and (ctx.submissions_dir / TEAM_AGENT_NAME).exists():
    try:
        smoke_rollout = collect_standalone_agent_rollout(
            ctx,
            TEAM_AGENT_NAME,
            steps=50,
            render=False,
            label="TEAMNAME_AGENT smoke rollout",
        )
        display(rollout_summary_table({TEAM_AGENT_NAME: smoke_rollout}))
        plot_reward_timeline(smoke_rollout, title="Submission smoke rollout reward timeline");
        plot_top_down_trajectory(smoke_rollout, title="Submission smoke rollout trajectory");
        plot_action_distribution(smoke_rollout, title="Submission smoke rollout action distribution");
    except Exception as exc:
        print("Tiny rollout failed:", type(exc).__name__, exc)
else:
    print("Tiny rollout skipped.")

Tiny rollout skipped.


In [9]:
if RUN_OPTIONAL_EVAL:
    try:
        from soccer_twos_project.evaluation import evaluate, safe_label, write_outputs
        rows, summary = evaluate(TEAM_AGENT_NAME, TEAM_AGENT_NAME, episodes=1, base_port=None)
        write_outputs(rows, summary, ctx.dirs["evals"], safe_label(TEAM_AGENT_NAME, TEAM_AGENT_NAME))
        print_json(summary)
    except Exception as exc:
        print("Optional eval failed:", type(exc).__name__, exc)
else:
    print("Optional eval skipped. Set RUN_OPTIONAL_EVAL=True for a one-episode self-check.")

Optional eval skipped. Set RUN_OPTIONAL_EVAL=True for a one-episode self-check.


## 7. Final Structural Readiness Summary

This is the final pass/fail message for the smoke submission artifact.

In [10]:
readiness = submission_readiness_summary(ctx, TEAM_AGENT_NAME, zip_path=final_zip)
if readiness["ready"]:
    print("STRUCTURALLY READY: TEAMNAME_AGENT.zip has the expected folder and required files, and import/action validation passed above if those cells succeeded.")
else:
    print("NOT READY YET: fix the missing checkpoint, export, package, or zip issue shown above.")

{
  "file_count": 10,
  "missing_required_files": [],
  "top_level": "test_1_AGENT",
  "zip_path": "/coc/scratch/vchopra/rl_training/project/soccer-twos-starter/artifacts/cs8803_soccer_twos/submissions/test_1_AGENT.zip"
}
{
  "folder_missing_required": [],
  "package_dir": "/coc/scratch/vchopra/rl_training/project/soccer-twos-starter/artifacts/cs8803_soccer_twos/submissions/test_1_AGENT",
  "package_exists": true,
  "ready": true,
  "team_agent_name": "test_1_AGENT",
  "zip_exists": true,
  "zip_manifest_ok": true,
  "zip_path": "/coc/scratch/vchopra/rl_training/project/soccer-twos-starter/artifacts/cs8803_soccer_twos/submissions/test_1_AGENT.zip"
}
STRUCTURALLY READY: TEAMNAME_AGENT.zip has the expected folder and required files, and import/action validation passed above if those cells succeeded.


## 8. Fresh-Clone Debug Commands (Recommended Before Gradescope)

Run these in a clean clone after copying only your `TEAMNAME_AGENT/` folder into the repo root.

In [11]:
final_folder_name = (Path(final_zip).with_suffix("").name if final_zip else "TEAMNAME_AGENT")
print("Fresh-clone sanity commands:")
print(f"python -m soccer_twos.watch -m {final_folder_name}")
print(f"python -m soccer_twos.watch -m1 {final_folder_name} -m2 ceia_baseline_agent")
print(f"python -m soccer_twos.watch -m1 {final_folder_name} -m2 example_team_agent")

Fresh-clone sanity commands:
python -m soccer_twos.watch -m test_1_AGENT
python -m soccer_twos.watch -m1 test_1_AGENT -m2 ceia_baseline_agent
python -m soccer_twos.watch -m1 test_1_AGENT -m2 example_team_agent


## Key Takeaways

This notebook validates the mechanics of submission: tiny training, checkpoint, export, final zip, manifest, import, `act()`, and optional rollout. It does not validate policy quality.

Submit the exact same `TEAMNAME_AGENT/` folder and `TEAMNAME_AGENT.zip` that pass your fresh-clone watch tests.

## What To Run Next

For the real project, run `03_full_training_pipeline.ipynb`, then `04_submission_and_report.ipynb` with your strongest trained agent and real team name.